In [1]:
# %%

import torch
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from datetime import datetime
import math
import torch_geometric as pyg
from torch_geometric.nn import GCNConv, MessagePassing

In [2]:
from Circuits import Circuits
circuits = Circuits()

Loading dataset files...


In [3]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # Shape: [1, max_len, d_model]

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class CustomGraphConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(CustomGraphConv, self).__init__(aggr='add')
        self.lin = nn.Linear(in_channels, out_channels)
        self.edge_weight = nn.Parameter(torch.ones(1))
        
    def forward(self, x, edge_index):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]
        
        # Transform node features
        x = self.lin(x)
        
        # Start propagating messages
        return self.propagate(edge_index, x=x)
    
    def message(self, x_j):
        # x_j has shape [E, out_channels]
        return x_j * self.edge_weight

class TextToGraphTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_heads, num_layers, max_seq_len=512, dropout=0.1):
        super(TextToGraphTransformer, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.positional_encoding = SinusoidalPositionalEncoding(embedding_dim, max_seq_len)

        # Transformer encoder for sequence processing
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=embedding_dim,
                nhead=num_heads,
                dim_feedforward=hidden_dim,
                dropout=dropout,
                batch_first=True
            ),
            num_layers=num_layers
        )
        
        # Graph neural network layers - using a simpler custom graph conv instead of TransformerConv
        self.graph_layers = nn.ModuleList([
            GCNConv(embedding_dim, embedding_dim),
            GCNConv(embedding_dim, embedding_dim)
        ])
        
        self.graph_ln1 = nn.LayerNorm(embedding_dim)
        self.graph_ln2 = nn.LayerNorm(embedding_dim)
        
        # Edge prediction MLP with improved design
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * embedding_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Optional node classifier
        self.node_classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, vocab_size)
        )
        
        # Structural bias module
        self.structural_bias = nn.Parameter(torch.zeros(1, 1, 1))
        
    def forward(self, input_seqs, seq_mask=None, return_node_logits=False):
        """
        input_seqs: [B, S]
        seq_mask: [B, S] (True for valid tokens, False for PAD tokens)
        return_node_logits: If True, returns node-level outputs for classification tasks
        """
        B, S = input_seqs.shape
        device = input_seqs.device
        
        # Embedding and positional encoding
        x = self.embedding(input_seqs)  # [B, S, D]
        x = self.positional_encoding(x)

        # Create attention mask: True for PAD tokens
        if seq_mask is None:
            attention_mask = (input_seqs == 0)  # [B, S]
        else:
            attention_mask = ~seq_mask
            
        # Transformer encoder
        x = self.transformer(x, src_key_padding_mask=attention_mask)  # [B, S, D]
        
        # Process each batch item with graph neural networks
        # We'll use a simplified approach here that's compatible with PyTorch Geometric
        enhanced_x = x.clone()
        
        # Create pairwise combinations for edges
        x_i = enhanced_x.unsqueeze(2).expand(-1, -1, S, -1)  # [B, S, S, D]
        x_j = enhanced_x.unsqueeze(1).expand(-1, S, -1, -1)  # [B, S, S, D]
        pairwise = torch.cat([x_i, x_j], dim=-1)   # [B, S, S, 2D]

        edge_logits = self.edge_mlp(pairwise).squeeze(-1)  # [B, S, S]
        
        # Apply padding mask
        if seq_mask is not None:
            # Create 2D mask: True for valid pairs, False for pairs involving padding
            mask_2d = seq_mask.unsqueeze(2) & seq_mask.unsqueeze(1)
            # Inverse for masking (True becomes padding position)
            inv_mask_2d = ~mask_2d
            # Apply mask (set padding positions to large negative value)
            edge_logits = edge_logits.masked_fill(inv_mask_2d, -1e4)
        
        # Make symmetric
        edge_logits = (edge_logits + edge_logits.transpose(1, 2)) / 2
        
        # Add structural bias to promote certain topological patterns
        edge_logits = edge_logits + self.structural_bias

        if return_node_logits:
            node_logits = self.node_classifier(enhanced_x)  # [B, S, vocab_size]
            return edge_logits, node_logits

        return edge_logits

In [4]:
# %%

def collate_fn(batch):
    seqs, mats = zip(*batch)

    # Convert sequences to torch tensors
    seqs = [torch.tensor(seq, dtype=torch.long) for seq in seqs]

    # Convert adjacency matrices (NumPy -> PyTorch)
    mats = [torch.tensor(mat, dtype=torch.float32) for mat in mats]

    # Get max sizes
    max_seq_len = max(len(seq) for seq in seqs)
    max_nodes = max(mat.size(0) for mat in mats)

    # Pad sequences
    padded_seqs = torch.stack([
        F.pad(seq, (0, max_seq_len - len(seq)), value=0)
        for seq in seqs
    ])

    # Pad adjacency matrices
    padded_mats = torch.stack([
        F.pad(mat, (0, max_nodes - mat.size(1), 0, max_nodes - mat.size(0)), value=0)
        for mat in mats
    ])

    seq_lengths = torch.tensor([len(seq) for seq in seqs])

    return padded_seqs, padded_mats, seq_lengths

In [5]:
# %%

dataset = list(zip(circuits.component_indices, circuits.graphs))
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

In [6]:
# %%

# Initialize model parameters
vocab_size = len(circuits.vocab)  # Number of unique components
embedding_dim = 128
hidden_dim = 256
num_heads = 8
num_layers = 4
dropout = 0.1

# Initialize the enhanced model
model = TextToGraphTransformer(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout
)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

TextToGraphTransformer(
  (embedding): Embedding(775, 128, padding_idx=0)
  (positional_encoding): SinusoidalPositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (graph_layers): ModuleList(
    (0-1): 2 x GCNConv(128, 128)
  )
  (graph_ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (graph_ln2): LayerNorm(

In [7]:
# %%

from torch.nn.utils.rnn import pad_sequence
from torch.amp import autocast, GradScaler  # Updated import for autocast

PAD_TOKEN_ID = 0
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler()  # For mixed-precision training

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

num_epochs = 200
dataloader = loader
save_every = 20  # Save every N epochs

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False)

    for input_seqs, adj_mats, seq_lengths in progress_bar:
        input_seqs = input_seqs.to(device, non_blocking=True)
        adj_mats = adj_mats.to(device, non_blocking=True)
        seq_lengths = seq_lengths.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):  # Updated autocast usage
            seq_mask = (input_seqs != PAD_TOKEN_ID)  # [B, S]
            predicted_logits = model(input_seqs, seq_mask)

            # Efficient masking (flatten first)
            mask = (input_seqs != PAD_TOKEN_ID)
            mask2d = mask.unsqueeze(2) & mask.unsqueeze(1)
            mask_flat = mask2d.view(-1)                   # [B*S*S]
            pred_flat = predicted_logits.view(-1)
            true_flat = adj_mats.view(-1)

            loss = criterion(pred_flat[mask_flat], true_flat[mask_flat])

        # Backprop with mixed precision
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.detach().item()  # Detach to avoid memory buildup
        progress_bar.set_postfix(loss=loss.item())

    scheduler.step()
    avg_loss = total_loss / len(dataloader)
    print(f"[Epoch {epoch+1}] Avg Loss: {avg_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")

    if epoch % save_every == 0 or epoch == num_epochs-1:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = f"Saves/EnhancedTtoGmodel_epoch{epoch}_{timestamp}.pth"
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'vocab_size': vocab_size,
            'embedding_dim': embedding_dim,
            'hidden_dim': hidden_dim,
            'num_heads': num_heads,
            'num_layers': num_layers,
            'dropout': dropout,
            'learning_rate': scheduler.get_last_lr()[0],
        }
        torch.save(checkpoint, save_path)
        print(f"💾 Checkpoint saved at epoch {epoch} → {save_path}")

# Save final model
torch.save(model.state_dict(), 'Saves/TextToGraphTransformer.pth')

[Epoch 1] Avg Loss: 0.3393, LR: 0.000030
💾 Checkpoint saved at epoch 0 → Saves/EnhancedTtoGmodel_epoch0_20250418_070935.pth


[Epoch 2] Avg Loss: 0.2572, LR: 0.000029


[Epoch 3] Avg Loss: 0.2554, LR: 0.000028


[Epoch 4] Avg Loss: 0.2544, LR: 0.000027


[Epoch 5] Avg Loss: 0.2532, LR: 0.000026


...

[Epoch 196] Avg Loss: 0.0724, LR: 0.000027


[Epoch 197] Avg Loss: 0.0720, LR: 0.000028


[Epoch 198] Avg Loss: 0.0721, LR: 0.000029


[Epoch 199] Avg Loss: 0.0709, LR: 0.000030


[Epoch 200] Avg Loss: 0.0713, LR: 0.000030
💾 Checkpoint saved at epoch 199 → Saves/EnhancedTtoGmodel_epoch199_20250418_072235.pth


In [8]:
# %%

def evaluate_adjacency_matrix(model, input_seq, vocab, device, threshold=0.5):
    model.eval()
    with torch.no_grad():
        input_tensor = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)  # [1, S]
        seq_len = input_tensor.size(1)
        seq_mask = (input_tensor != PAD_TOKEN_ID)
        
        # Pass input to model
        logits = model(input_tensor, seq_mask)
        
        probs = torch.sigmoid(logits.squeeze(0))  # Shape: (S, S)
        binary_adj = (probs > threshold).float()

        print("\nPredicted Adjacency Matrix (Binary, N x N):")
        print(binary_adj.cpu().numpy())

In [9]:
# %%

def load_checkpoint(path, device='cuda'):
    checkpoint = torch.load(path, map_location=device)

    model = TextToGraphTransformer(
        vocab_size=checkpoint['vocab_size'],
        embedding_dim=checkpoint['embedding_dim'],
        hidden_dim=checkpoint['hidden_dim'],
        num_heads=checkpoint['num_heads'],
        num_layers=checkpoint['num_layers'],
        dropout=checkpoint['dropout']
    ).to(device)

    model.load_state_dict(checkpoint['model_state_dict'])

    optimizer = torch.optim.AdamW(model.parameters(), lr=checkpoint['learning_rate'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    if 'scheduler_state_dict' in checkpoint:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print(f"✅ Loaded model from {path} (epoch {checkpoint['epoch']}, lr={checkpoint['learning_rate']:.6f})")
        return model, optimizer, scheduler, checkpoint['epoch']
    else:
        print(f"✅ Loaded model from {path} (epoch {checkpoint['epoch']})")
        return model, optimizer, None, checkpoint['epoch']

In [13]:
file_name = "Saves/EnhancedTtoGmodel_epoch199_20250418_072235.pth"
checkpoint_path = file_name # Replace with your file

# 2. Load the model and optimizer
model, optimizer, scheduler, start_epoch = load_checkpoint(checkpoint_path, device=device)


ex_index = 320
sample_input = circuits.component_indices[ex_index]
print("Sample Input Sequence:", circuits.component_lists[ex_index])
print("Actual Adjacency Matrix:")
print(circuits.graphs[ex_index])
evaluate_adjacency_matrix(model, sample_input, circuits.vocab, device)

✅ Loaded model from Saves/EnhancedTtoGmodel_epoch199_20250418_072235.pth (epoch 199, lr=0.000030)
Sample Input Sequence: ['VSS', 'VIN1', 'VB1', 'IB1', 'IB2', 'IB3', 'IB4', 'VLO1', 'VLO2', 'VLO3', 'VLO4', 'VBB1', 'VBB2', 'VBB3', 'VBB4', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'PM2', 'PM2_D', 'PM2_G', 'PM2_S', 'PM2_B', 'PM3', 'PM3_D', 'PM3_G', 'PM3_S', 'PM3_B', 'PM4', 'PM4_D', 'PM4_G', 'PM4_S', 'PM4_B', 'PM5', 'PM5_D', 'PM5_G', 'PM5_S', 'PM5_B', 'PM6', 'PM6_D', 'PM6_G', 'PM6_S', 'PM6_B', 'R1', 'R1_P', 'R1_N', 'R2', 'R2_P', 'R2_N', 'R3', 'R3_P', 'R3_N', 'R4', 'R4_P', 'R4_N', 'R5', 'R5_P', 'R5_N', 'R6', 'R6_P', 'R6_N', 'R7', 'R7_P', 'R7_N', 'R8', 'R8_P', 'R8_N', 'C1', 'C1_P', 'C1_N', 'C2', 'C2_P', 'C2_N', 'C3', 'C3_P', 'C3_N', 'C4', 'C4_P', 'C4_N', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'NM2', 'NM2_D', 'NM2_G', 'NM2_S', 'NM2_B', 'NM3', 'NM3_D', 'NM3_G', 'NM3_S', 'NM3_B', 'NM4', 'NM4_D', 'NM4_G', 'NM4_S', 'NM4_B', 'NM5', 'NM5_D', 'NM5_G', 'NM5_S', 'NM5_B']
Actual Adjacency Matrix:
[[